# Running an Optimiser: A Measured Before/After [Agent Patterns - Module 09]

> **MLCourse - Agentic AI - Agent Patterns**

This is the notebook the module exists for. We take the exact program from
notebook 02, run a DSPy optimiser over the 8 training examples, and re-score
the same held-out dev set with the same metric.

Same model. Same signature. Same data. The only thing that changes is the
prompt text DSPy generates - and it generates it from *your examples*
instead of from your intuition.

### What you will learn

1. What `BootstrapFewShot` actually does (it is simpler than it sounds).
2. How to compile a program and re-evaluate it honestly.
3. How to read the optimised prompt DSPy produced.
4. When optimisation is worth the tokens - and when it is not.

### Key takeaways

- Optimising = searching for demonstrations/instructions that maximise
  your metric. It is not magic and it is not fine-tuning.
- The before/after number is the deliverable. Report it even if it is flat.

### Setup: imports, environment, track discovery


In [ ]:
import os
import time
import random
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives INSIDE the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)

GROQ_API_KEY = os.environ["GROQ_API_KEY"]   # loud failure if missing, by design
MODEL = "qwen/qwen3.8-27b"                   # Groq-hosted; never OpenAI

print(f"Track root : {TRACK}")
print(f"Model      : {MODEL} (via Groq)")
print(f"Key loaded : {bool(GROQ_API_KEY)}")


### Point DSPy at Groq (through LiteLLM)


In [ ]:
# dspy.LM is a thin wrapper over LiteLLM. The "groq/" prefix is the LiteLLM
# provider route - the rest is the Groq model id.

import dspy

lm = dspy.LM(
    f"groq/{MODEL}",
    api_key=GROQ_API_KEY,
    temperature=0.0,      # deterministic-ish: we are going to MEASURE things
    max_tokens=700,
    num_retries=5,        # LiteLLM backs off on 429 (Groq free tier = 8000 TPM)
)
dspy.configure(lm=lm)

print("DSPy configured.")
print("dspy version:", dspy.__version__)


### 1. Rebuild the task

Notebooks are independent, so we restate the signature, data and metric.
This is deliberately identical to notebook 02 - the comparison is only
meaningful if nothing else moved.

### Task, data and metric (identical to notebook 02)


In [ ]:
from typing import Literal

class Triage(dspy.Signature):
    """Assign a support priority to an incoming customer message."""
    message: str = dspy.InputField(desc="the raw customer message")
    priority: Literal["P0", "P1", "P2"] = dspy.OutputField(desc="the triage priority")

TRAIN = [
    ("I cannot log in at all, it says account locked since this morning.", "P0"),
    ("You charged my card twice for order 8812, please refund one.",       "P1"),
    ("How do I change the language in the settings screen?",               "P2"),
    ("The whole dashboard is blank for everyone on my team since 9am.",    "P0"),
    ("My subscription renewed at the old price, the invoice looks wrong.", "P1"),
    ("It would be nice if the export button remembered my last folder.",   "P2"),
    ("App crashes immediately on launch after the update, unusable.",      "P0"),
    ("Payment failed three times but the money left my account.",          "P1"),
    ("URGENT!! Add a dark mode toggle immediately, this is top priority!", "P2"),
    ("No hurry: the annual plan charged me the monthly rate by mistake.",  "P1"),
]
DEV = [
    ("Nobody in the office can open the site, it times out.",              "P0"),
    ("I was billed for two seats but we only ever had one.",               "P1"),
    ("Where can I find the keyboard shortcuts list?",                      "P2"),
    ("Everything I typed was lost when the editor froze and reset.",       "P0"),
    ("Please cancel my plan and refund the last charge.",                  "P1"),
    ("The dark theme colours are a bit low contrast, just feedback.",      "P2"),
    ("Login page returns a 500 error for all our users.",                  "P0"),
    ("An unexpected 49 rupee fee showed up on this month statement.",      "P1"),
    ("URGENT!!! You MUST add a bulk delete button, this is critical!!",    "P2"),
    ("No rush at all, but the invoice total is 200 more than quoted.",     "P1"),
    ("I am furious, the icons are ugly since the redesign.",               "P2"),
    ("Server returns 503 to every request, our shop is offline.",          "P0"),
]

trainset = [dspy.Example(message=m, priority=p).with_inputs("message") for m, p in TRAIN]
devset = [dspy.Example(message=m, priority=p).with_inputs("message") for m, p in DEV]

def priority_match(example, pred, trace=None):
    return float(str(pred.priority).strip().upper() == example.priority)

def evaluate(program, dataset, metric, pause=1.5, label=""):
    rows, correct = [], 0.0
    for i, ex in enumerate(dataset, 1):
        for attempt in range(5):
            try:
                pred = program(**ex.inputs()); break
            except Exception as e:
                wait = 2 ** attempt + random.random()
                print(f"  retry {attempt+1} in {wait:.1f}s ({type(e).__name__})")
                time.sleep(wait)
        else:
            raise RuntimeError("giving up after 5 retries")
        s = metric(ex, pred); correct += s
        rows.append((ex.message, ex.priority, str(pred.priority), s))
        print(f"  [{i}/{len(dataset)}] gold={ex.priority} pred={pred.priority} {'OK' if s else 'MISS'}")
        time.sleep(pause)
    acc = correct / len(dataset)
    print(f"\n{label} accuracy: {acc:.1%}  ({int(correct)}/{len(dataset)})")
    return acc, rows

print(f"train={len(trainset)} dev={len(devset)}")


### BEFORE


In [ ]:
baseline_program = dspy.Predict(Triage)

print("=== BEFORE: zero-shot ===")
before_acc, before_rows = evaluate(baseline_program, devset, priority_match, label="BEFORE")


### 2. What `BootstrapFewShot` does

Do not be intimidated by the name. The algorithm is four steps:

1. Run the current (unoptimised) program on training examples.
2. Keep only the runs where the **metric passed** - i.e. the program
   happened to get it right.
3. Those successful input/output traces become **demonstrations**.
4. Paste those demonstrations into the prompt.

That is it. It is "self-generated few-shot examples, filtered by your
metric". `max_labeled_demos` additionally allows using your gold-labelled
rows directly as demos, which is what rescues the case where the zero-shot
program gets almost everything wrong.

We pick this optimiser because it is the **cheapest**. `MIPROv2` and
`BootstrapFewShotWithRandomSearch` search over many candidate prompts and
will happily burn tens of thousands of tokens - a bad fit for a free tier
and an unnecessary complication for a first look.

### Compile


In [ ]:
from dspy.teleprompt import BootstrapFewShot

optimizer = BootstrapFewShot(
    metric=priority_match,
    max_bootstrapped_demos=3,   # self-generated, metric-verified demos
    max_labeled_demos=4,        # gold rows used directly as demos
    max_rounds=1,               # one pass: keep the token bill small
)

print("Compiling (this makes real Groq calls over the trainset)...")
t0 = time.time()
optimized_program = optimizer.compile(dspy.Predict(Triage), trainset=trainset)
print(f"Compiled in {time.time() - t0:.0f}s")
print("demos attached:", len(optimized_program.predict.demos)
      if hasattr(optimized_program, "predict") else len(optimized_program.demos))


### AFTER


In [ ]:
print("=== AFTER: optimised ===")
after_acc, after_rows = evaluate(optimized_program, devset, priority_match, label="AFTER")


### The headline number


In [ ]:
n = len(devset)
print("=" * 58)
print(f"{'':12s}{'accuracy':>12s}{'correct':>12s}")
print("-" * 58)
print(f"{'BEFORE':12s}{before_acc:>11.1%}{int(before_acc*n):>9d}/{n}")
print(f"{'AFTER':12s}{after_acc:>11.1%}{int(after_acc*n):>9d}/{n}")
print("-" * 58)
delta = after_acc - before_acc
print(f"{'DELTA':12s}{delta:>+11.1%}{int(round(delta*n)):>+9d} examples")
print("=" * 58)

if delta > 0:
    print("\nOptimisation helped: the demos taught the house rubric.")
elif delta == 0:
    print("\nNo change. Honest result - report it, do not hide it.")
else:
    print("\nIt got WORSE. This happens on tiny dev sets; see the notes below.")


### Row-by-row diff


In [ ]:
print(f"{'gold':>5}  {'before':>6}  {'after':>6}   message")
print("-" * 78)
for (msg, gold, b, _), (_, _, a, _) in zip(before_rows, after_rows):
    flag = "  <-- fixed" if (b != gold and a == gold) else ("  <-- broke" if (b == gold and a != gold) else "")
    print(f"{gold:>5}  {b:>6}  {a:>6}   {msg[:44]}{flag}")


### 3. Read the prompt the optimiser wrote

This is the part people skip and then wonder what DSPy "did". The compiled
program is not a black box - it is a prompt, and you can read it.

### Inspect the compiled prompt


In [ ]:
dspy.inspect_history(n=1)


### The demos, listed explicitly


In [ ]:
demos = getattr(optimized_program, "demos", None)
if demos is None:
    demos = optimized_program.predict.demos

print(f"{len(demos)} demonstration(s) baked into the prompt:\n")
for i, d in enumerate(demos, 1):
    dd = d if isinstance(d, dict) else d.toDict()
    print(f"  {i}. [{dd.get('priority')}] {str(dd.get('message'))[:70]}")


### Save the compiled program


In [ ]:
# The artifact is JSON: signature + demos. It is small, diffable, and belongs
# in version control next to your code.

optimized_program.save("triage_optimized.json")
print("saved -> triage_optimized.json")
print(Path("triage_optimized.json").read_text(encoding="utf-8")[:600], "...")


### 4. When is this worth it?

**Worth optimising when:**

- The task has a **house convention** the model cannot infer (our rubric,
  your label taxonomy, your output schema, your tone rules).
- You have, or can cheaply write, 10-50 labelled examples.
- You have a metric you actually trust.
- The prompt will be called many times, so a one-off compile cost amortises.

**Not worth it when:**

- The task is already at ceiling zero-shot. Measure first.
- Your metric is vague ("is it good?"). You will optimise the metric's
  blind spots, not the task.
- You have three examples. There is nothing to learn from.
- The call runs once. Compiling costs more than the call.

### Pitfalls recap

- **Tiny dev sets are noisy.** 12 rows means one flip is ~8 points. Treat
  a +1-example delta as noise; look for the pattern in the row diff.
- **Optimising on the dev set is cheating.** Compile on `trainset`, score
  on `devset`. Never let the optimiser see the rows you report on.
- **Expensive optimisers bite.** `MIPROv2` can make hundreds of calls.
  Start with `BootstrapFewShot`, and read the docs on token cost before
  upgrading.
- **Re-compile after a model swap.** The demos were selected against one
  model's behaviour.

### Next

Notebook 04 turns the compiled artifact into something you would actually
ship, and covers the operational side: versioning, drift, and cost.